# Imports


In [ ]:
# Import required libraries
import polars as pl
from pathlib import Path

# Data Path


In [ ]:
# Define data folder path
data_folder = Path("../data")

# Load Dataset


In [ ]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview data
sales_df.head()

## Festival vs Non-Festival Sales


In [ ]:
# Revenue comparison between festival and normal days
festival_sales = sales_df.group_by("festival").agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue"),

    pl.col("revenue").mean().round().alias("avg_order_value")
)

festival_sales.sort("total_revenue", descending=True)

## Festival Revenue Share


In [ ]:
festival_sales = festival_sales.with_columns(
    (
        pl.col("total_revenue") / pl.col("total_revenue").sum() * 100
    ).round(1).alias("revenue_share_pct")
)

festival_sales

## Festival Category Performance


In [ ]:
festival_category = sales_df.group_by(
    ["festival", "category"]
).agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue")
).sort(
    ["festival", "total_revenue"],
    descending=[False, True]
)

festival_category

# City Festival Performance


In [ ]:
festival_city = sales_df.group_by(
    ["festival", "city"]
).agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue")
).sort(
    ["festival", "total_revenue"],
    descending=[False, True]
)

festival_city.head(10)

In [ ]:
import polars as pl

# Aggregate sales by city
city_summary = (
    sales_df.group_by("city")
    .agg([
        pl.count("transaction_id").alias(
            "transactions"),  # total transactions per city
        # total revenue per city
        pl.sum("revenue").alias("total_revenue")
    ])
    .with_columns(
        # average revenue per transaction
        (pl.col("total_revenue") / pl.col("transactions")
         ).round().alias("avg_revenue_per_txn")
    )
)

city_summary.sort("avg_revenue_per_txn", descending=True)

## Save the Data set


In [ ]:
festival_sales.write_parquet(data_folder/"festival_sales.parquet")

In [ ]:
festival_sales